In [1]:
from pathlib import Path
import pandas as pd
import spatialdata as sd

import pickle
DATA_DIR = Path("/root/capsule/data")
SCRATCH_DIR = Path("/root/capsule/scratch")
MATCH_DIR = SCRATCH_DIR / "ophys_transcriptomics_match_tables"
TRANSCRIPT_DIR = SCRATCH_DIR / 'transcriptomics'
TRANSCRIPT_DIR.mkdir(exist_ok=True)

/opt/conda/envs/xenium_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


load data information

In [ ]:
data_info = pickle.load(open('data_info_hcr.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [755252, 767018, 767022, 782149, 788406, 790322]


In [3]:
for i_mouse, mouse_id in enumerate(mouse_ids):
    hcr_processed_path = DATA_DIR / data_info[mouse_id]['hcr']['processed']

    output_path = TRANSCRIPT_DIR / f'{mouse_id}h_cellxgene.csv'

    if 'pairwise_unmixing' in [f.name for f in hcr_processed_path.glob('*')]:
        cellxgene_path = hcr_processed_path / 'pairwise_unmixing' / 'all_cells_unmixed' / 'unmixed_all_cells.csv'
    else:
        cellxgene_path = hcr_processed_path / 'all_cells_unmixed' / 'unmixed_all_cells.csv'

    cellxgene_df = pd.read_csv(cellxgene_path)

    cellxgene_df = cellxgene_df.rename(columns={col: col.split('-')[-1] for col in cellxgene_df.columns if col.startswith('R')})
    cellxgene_df = cellxgene_df.rename(columns={'cell_id': 'cell_id - hcr'})
    cellxgene_df.insert(0, 'mouse_id', [mouse_id]*len(cellxgene_df))

    cellxgene_df.to_csv(output_path, index=False)

    output_path = TRANSCRIPT_DIR / f'{mouse_id}h_cell_types.csv'
    cell_types_df = pd.DataFrame()
    hcr_mapped_path = DATA_DIR / data_info[mouse_id]['hcr']['cell_types']
   

    cell_types_df = pd.read_csv(hcr_mapped_path / f'unmixed_all_cells_flat_mapping/mapped_data/basic_results.csv', skiprows=4)
    cell_types_df['cell_id'] = cellxgene_df['cell_id - hcr'].copy()
    cell_types_df['mouse_id'] = mouse_id
    cell_types_df = cell_types_df.rename(columns={'cell_id': 'cell_id - hcr'})
    cell_types_df.to_csv(output_path, index=False)


In [5]:
main_subclasses = ["046 Vip Gaba",
"052 Pvalb Gaba",
"051 Pvalb chandelier Gaba",
"053 Sst Gaba",
"049 Lamp5 Gaba",
"047 Sncg Gaba" ] 
for i_mouse, mouse_id in enumerate(mouse_ids):
    ophys_hcr_match = pd.read_csv(MATCH_DIR / f'{mouse_id}h_ophys_transcriptomics_match.csv')
    cell_types_df = pd.read_csv(TRANSCRIPT_DIR / f'{mouse_id}h_cell_types.csv')
    cell_types_coreg_df = ophys_hcr_match.merge(cell_types_df, on = ['mouse_id','cell_id - hcr'], how = 'left')
    cell_types_coreg_df = cell_types_coreg_df[cell_types_coreg_df['subclass_name'].isin(main_subclasses)]
    cell_types_coreg_df.to_csv(TRANSCRIPT_DIR / f'{mouse_id}h_cell_types_coreg.csv',index=False)

    cellxgene_df = pd.read_csv(TRANSCRIPT_DIR / f'{mouse_id}h_cellxgene.csv')
    cellxgene_coreg_df = ophys_hcr_match.merge(cellxgene_df, on = ['mouse_id','cell_id - hcr'], how = 'left')
    cellxgene_coreg_df = cellxgene_coreg_df[cellxgene_coreg_df['unique_id'].isin(cell_types_coreg_df['unique_id'].values)]
    cellxgene_coreg_df = cellxgene_coreg_df[[col for col in cellxgene_coreg_df.columns if 'Code' not in col and 'Control' not in col]]
    cellxgene_coreg_df.to_csv(TRANSCRIPT_DIR / f'{mouse_id}h_cellxgene_coreg.csv',index=False)
    print(mouse_id, len(cellxgene_coreg_df), 'done')


755252 190 done
767018 105 done
767022 244 done
782149 119 done
788406 192 done
790322 159 done


In [6]:
for i_mouse, mouse_id in enumerate(mouse_ids):
    cell_types_coreg_df = pd.read_csv(TRANSCRIPT_DIR / f'{mouse_id}h_cell_types_coreg.csv')
    print(f"{mouse_id} cell type distribution:")
    print(cell_types_coreg_df['subclass_name'].value_counts())
    print()

755252 cell type distribution:
subclass_name
046 Vip Gaba                 68
052 Pvalb Gaba               48
049 Lamp5 Gaba               40
053 Sst Gaba                 15
051 Pvalb chandelier Gaba    12
047 Sncg Gaba                 7
Name: count, dtype: int64

767018 cell type distribution:
subclass_name
046 Vip Gaba      66
052 Pvalb Gaba    24
053 Sst Gaba      12
049 Lamp5 Gaba     3
Name: count, dtype: int64

767022 cell type distribution:
subclass_name
046 Vip Gaba                 91
052 Pvalb Gaba               67
049 Lamp5 Gaba               45
053 Sst Gaba                 17
051 Pvalb chandelier Gaba    17
047 Sncg Gaba                 7
Name: count, dtype: int64

782149 cell type distribution:
subclass_name
046 Vip Gaba                 62
049 Lamp5 Gaba               41
052 Pvalb Gaba                8
053 Sst Gaba                  4
051 Pvalb chandelier Gaba     3
047 Sncg Gaba                 1
Name: count, dtype: int64

788406 cell type distribution:
subclass_name
052 Pva